In [9]:
%pip install faiss-cpu

# 1. Dense Vector Search (밀집 벡터 검색)


$$
\text{similarity}(q, d) = \cos(\theta)
= \frac{q \cdot d}{\|q\| \times \|d\|}
$$


- 원리 : 문장을 고차원 벡터로 변환하고 코사인 유사도를 이용해서 의미적 유사성 측정
- 장점
  - 의미적 유사성 포착(동의어)
  - 비가온다 <->장마가 시작됐다
- 단점
  - 고유명사, 숫자 등 정확한 매칭에 약해

# 2. BM25 (Best Match 25)
  - 키워드 기반 희소 검색

$$
\text{BM25}(D, Q) = \sum_{q_i \in Q} \text{IDF}(q_i) \cdot
\frac{f(q_i, D)(k_1 + 1)}
{f(q_i, D) + k_1\left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}
$$


**IDF(qᵢ)**: 단어 qᵢ의 역문서 빈도 (드문 단어일수록 값이 큼)  
**f(qᵢ, D)**: 문서 D에서 단어 qᵢ가 등장한 횟수  
**k₁**: 포화 파라미터 (일반적으로 1.2 ~ 2.0)  
**b**: 문서 길이 정규화 파라미터 (일반적으로 0.75)  
**avgdl**: 전체 문서들의 평균 길이


- TF-IDF 개선버전, 문서 내 단어 빈도와 희소성을 고려
- 장점
  - 정확한 키워드 매칭
  - 고유명사 검색에 강함
- 단점
  - 의미적 유사성 포착 불가
  - 비<->장마 연결 어려움  

# 3. Reciprocal Rank Fusion (RRF)

$$
\text{RRF}(d) = \sum_i \frac{1}{k + \text{rank}_i(d)}
$$

**d**: 문서  
**k**: 상수 (일반적으로 60 사용)  
**rankᵢ(d)**: i번째 검색 시스템에서 문서 d가 받은 순위



- 여러검색 결과의 순위를 통합하여 최종 랭킹을 생성
- 장점
  - 다양한 검색 방식의 장점 결합
  - 점수 스케일이 다른 검색 결과도 통합 가능

In [10]:
# 문장 임베딩으로 벡터 검색
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [12]:
# 한국어 임베딩 모델 로드
model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')
# sample doc
documents = [
    "올해 여름 장마가 시작됐다",
    "프린스턴 대학교에서 학위를 받았다",
    "갤럭시 S5가 출시됐다"
]
# 문서 임베딩
doc_embeddings = model.encode(documents)

# FAISS 인덱스 생성
dimension =  doc_embeddings.shape[1]   #(3, 768)
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings.astype('float32'))

# 쿼리 검색
query = '비가 많이 올 시기는?'
query_embedding =  model.encode([query])
distance, indices =  index.search( query_embedding.astype('float32'), k=3)
print('검색결과')
for i, (dist,idx) in  enumerate( zip(distance[0], indices[0]) ):
  print(f' {i+1}. {documents[idx]} ( 거리 : {dist:.4f} )')

검색결과
 1. 올해 여름 장마가 시작됐다 ( 거리 : 391.9998 )
 2. 갤럭시 S5가 출시됐다 ( 거리 : 702.7926 )
 3. 프린스턴 대학교에서 학위를 받았다 ( 거리 : 717.1246 )
